# G03 - Zaman Serisi Regresyon Modellemesi ve SHAP Açıklanabilirlik Analizi
## Hazırlayan: G03 Proje Ekibi

Bu çalışma defteri;
1. `ulasim_toplanmis_ozet.csv` verisinin zamana göre Train-Test ayrımını,
2. Naif Baseline modeli (geçen hafta aynı saatteki ortalama),
3. Üç farklı regresyon modelini (Doğrusal Regresyon, Random Forest, HistGradientBoosting),
4. MAE ve RMSE kıyaslama tablosunu,
5. En iyi modelin `model.joblib` olarak kaydedilmesini,
6. SHAP analizi ile hava durumu ve takvim etkilerinin açıklanmasını içerir.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import shap

# 1. Veri Yükleme ve Zaman Sıralaması
df = pd.read_csv('../01_Veri/ulasim_toplanmis_ozet.csv')
df['tarih_saat'] = pd.to_datetime(df['tarih']) + pd.to_timedelta(df['saat'], unit='h')
df = df.sort_values('tarih_saat').reset_index(drop=True)

feats = ['saat', 'haftanin_gunu', 'hafta_sonu', 'tatil', 'okul_donemi', 'sicaklik', 'yagis', 'ruzgar']
hedef = 'yolcu'
df.head()

In [ ]:
# 2. Zamana Göre Ayrım (Son 14 gün test kümesi - Kesinlikle shuffle yok)
cutoff = df['tarih_saat'].max() - pd.Timedelta(days=14)
train = df[df['tarih_saat'] < cutoff].copy()
test = df[df['tarih_saat'] >= cutoff].copy()

print(f"Eğitim Seti: {len(train):,} satır | Test Seti: {len(test):,} satır")

In [ ]:
# 3. Naif Baseline Model (Geçen hafta aynı gün ve aynı saatteki talep ortalaması)
baseline_map = train.groupby(['saat', 'haftanin_gunu'])[hedef].mean()
test['baseline_pred'] = test.set_index(['saat', 'haftanin_gunu']).index.map(baseline_map).fillna(train[hedef].mean())

sonuclar = [{
    'Model': 'Naif Baseline',
    'MAE': mean_absolute_error(test[hedef], test['baseline_pred']),
    'RMSE': np.sqrt(mean_squared_error(test[hedef], test['baseline_pred']))
}]
pd.DataFrame(sonuclar)

In [ ]:
# 4. Üç Regresyon Modelinin Eğitimi ve Karşılaştırılması
modeller = {
    'Doğrusal Regresyon': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=150, random_state=42)
}

egitilmis = {}
for ad, m in modeller.items():
    m.fit(train[feats], train[hedef])
    p = m.predict(test[feats])
    sonuclar.append({
        'Model': ad,
        'MAE': mean_absolute_error(test[hedef], p),
        'RMSE': np.sqrt(mean_squared_error(test[hedef], p))
    })
    egitilmis[ad] = m

df_sonuc = pd.DataFrame(sonuclar)
print('========== MODEL KIYASLAMA TABLOSU ==========')
display(df_sonuc)

In [ ]:
# 5. En Başarılı Modelin Kaydedilmesi (model.joblib)
best = egitilmis['HistGradientBoosting']
joblib.dump(best, '../03_Model/model.joblib')
print('En iyi model başarıyla kaydedildi: 03_Model/model.joblib')

In [ ]:
# 6. SHAP Açıklanabilirlik Analizi (Hava Durumu ve Saat Etkisi)
test_ornek = test[feats].sample(n=min(1000, len(test)), random_state=42)
explainer = shap.Explainer(best, train[feats].sample(n=500, random_state=42))
shap_values = explainer(test_ornek)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, show=True)